# PPMI RNA-seq featureCounts 기반 차등발현분석 비교 노트북  
**Python 3.9.21 / PyDESeq2 0.4.12 기준**

이 노트북은 PPMI IR3 RNA-seq `featureCounts` 파일과 IR3 메타데이터를 이용해 다음을 수행합니다.

1. **전체 데이터셋(full dataset)** 에 대해 차등발현분석(DE)
2. **Stratified K-fold** 분할
3. 각 fold마다
   - **train subset** 에 대해 DE
   - **test subset** 에 대해 DE
4. full 결과와 각 fold 결과를 비교
   - 유의 DEG 수
   - full 결과와의 overlap / Jaccard
   - log2FC 상관
   - 방향성 일치율
5. fold 반복 전체에서의 gene-level 안정성 요약

---

## 전제
- 입력은 **샘플별 개별 featureCounts txt 파일들**입니다.
- 파일명은 README의 IR3 naming convention을 따릅니다.
  - `FIELD 1`: study/release
  - `FIELD 2`: PATNO
  - `FIELD 3`: visit
  - `FIELD 4`: sampleID
  - `FIELD 5`: sequencing facility ID
  - counts 예시는 `PPMI-Phase1-IR3.3056.BL.PP0016-0581.5104-SL-0620.featureCounts.GencodeV29.txt` 형태입니다.
- README에 따르면 IR3 counts는 `featureCounts v1.6.3`, annotation은 `gencode.v29.primary_assembly.annotation.gtf`를 사용했습니다.
- 메타데이터는 보통 `metaDataIR3.tsv` 형태이며, `DIAGNOSIS`, `CLINICAL_EVENT`, `QCflagIR3`, `Sample`, `HudAlphaID` 같은 컬럼을 포함한다고 가정합니다.

> **주의**  
> fold의 **test subset** 에 대해 DE를 수행하는 것은 가능하지만, 일반적인 예측 모델 평가와는 달리 **표본 수 감소로 인해 결과가 불안정**할 수 있습니다.  
> 이 노트북의 목적은 “train/test 분할 시 DE 결과가 얼마나 흔들리는지”를 보는 것입니다.


In [ ]:
# 필요 시 최초 1회 실행
# %pip install -U pydeseq2==0.4.12 scikit-learn pandas numpy matplotlib jupyter


In [ ]:
from __future__ import annotations

import json
import math
import warnings
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
plt.rcParams["figure.figsize"] = (8, 4)


In [ ]:
@dataclass
class AnalysisConfig:
    # 경로
    COUNTS_DIR: str = "./Dataset/RNA/IR3/counts/"
    METADATA_PATH: str = "./Dataset/RNA/IR3/metaDataIR3.csv"
    RESULTS_DIR: str = "./de_stability_results"

    # 파일 탐색
    COUNT_FILE_GLOB: str = "*featureCounts*"
    RECURSIVE_SEARCH: bool = False

    # 메타데이터 필터
    DIAGNOSES: tuple = ("Genetic Cohort", "Control")
    VISITS: tuple = ("BL",)
    QC_PASS_ONLY: bool = True

    # 실험 설계
    CASE_LABEL: str = "Genetic Cohort"    #"PD"
    CONTROL_LABEL: str = "Control"
    CONDITION_COL: str = "condition"

    # 선택적 공변량
    CATEGORICAL_COVARIATES: tuple = ()
    CONTINUOUS_COVARIATES: tuple = ()

    # K-fold
    N_SPLITS: int = 5
    SHUFFLE: bool = True
    RANDOM_STATE: int = 42

    # DE 파라미터
    MIN_TOTAL_COUNT_PER_GENE: int = 10
    ALPHA: float = 0.05
    SIGNIFICANCE_METRIC: str = "padj"   # "padj" or "pvalue"

    FIT_TYPE: str = "parametric"
    REFIT_COOKS: bool = True
    MIN_REPLICATES: int = 3
    QUIET: bool = True
    LOW_MEMORY: bool = False
    N_CPUS: Optional[int] = None
    COOKS_FILTER: bool = True
    INDEPENDENT_FILTER: bool = True

    # subset 유효성 검사
    MIN_SAMPLES_PER_GROUP_FOR_DE: int = 2

cfg = AnalysisConfig()
Path(cfg.RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


In [ ]:
def read_metadata_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Metadata file not found: {path}")

    # 확장자 기반 기본 처리
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t")

    # 자동 구분자 추정
    try:
        return pd.read_csv(path, sep=None, engine="python")
    except Exception:
        return pd.read_csv(path)

meta_raw = read_metadata_table(cfg.METADATA_PATH)
print("metadata shape:", meta_raw.shape)
print("metadata columns:")
print(meta_raw.columns.tolist())

preview_cols = [c for c in ["PATNO", "DIAGNOSIS", "CLINICAL_EVENT", "QCflagIR3", "HudAlphaID", "Sample", "GENDER", "SEX", "AGE"] if c in meta_raw.columns]
display(meta_raw[preview_cols].head())


In [ ]:
FEATURECOUNTS_SUFFIX_PATTERNS = [
    r"\.featureCounts\.GencodeV29\.txt$",
    r"\.featureCounts\.GencodeV29$",
    r"\.featureCounts-GencodeV29\.txt$",
    r"\.featureCounts-GencodeV29$",
    r"\.featureCounts.*$",
]

def strip_featurecounts_suffix(name: str) -> str:
    out = name
    for pat in FEATURECOUNTS_SUFFIX_PATTERNS:
        out = re.sub(pat, "", out)
    return out

def normalize_sample_key(x: object) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = Path(s).name
    s = re.sub(r"\.txt$", "", s)
    s = strip_featurecounts_suffix(s)
    return s

def parse_count_filename(path: str | Path) -> Dict[str, str]:
    name = Path(path).name
    sample_key = normalize_sample_key(name)
    parts = sample_key.split(".")
    return {
        "file_name": name,
        "sample_key": sample_key,
        "study_release": parts[0] if len(parts) > 0 else "",
        "PATNO_file": parts[1] if len(parts) > 1 else "",
        "CLINICAL_EVENT_file": parts[2] if len(parts) > 2 else "",
        "SampleID_file": parts[3] if len(parts) > 3 else "",
        "HudAlphaID_file": parts[4] if len(parts) > 4 else "",
    }

def read_featurecounts_file(path: str | Path) -> pd.Series:
    path = Path(path)
    df = pd.read_csv(path, sep="\t", comment="#", low_memory=False)
    if df.empty:
        raise ValueError(f"Empty featureCounts file: {path}")

    possible_gene_cols = ["Geneid", "gene_id", "GeneID", "gene"]
    gene_col = next((c for c in possible_gene_cols if c in df.columns), df.columns[0])
    count_col = df.columns[-1]

    counts = pd.Series(
        pd.to_numeric(df[count_col], errors="coerce").fillna(0).astype(int).values,
        index=df[gene_col].astype(str).values,
        name=normalize_sample_key(path.name),
    )
    return counts

def build_counts_matrix(counts_dir: str | Path, file_glob: str = "*featureCounts*", recursive: bool = False):
    counts_dir = Path(counts_dir)
    files = sorted(counts_dir.rglob(file_glob) if recursive else counts_dir.glob(file_glob))
    if not files:
        raise FileNotFoundError(
            f"No files matched {file_glob!r} under {counts_dir}. "
            "Check cfg.COUNTS_DIR / cfg.COUNT_FILE_GLOB / cfg.RECURSIVE_SEARCH."
        )

    sample_series = []
    file_records = []

    for fp in files:
        sample_series.append(read_featurecounts_file(fp))
        file_records.append(parse_count_filename(fp))

    counts_df = pd.DataFrame(sample_series)
    counts_df.index = [s.name for s in sample_series]
    counts_df.index.name = "sample_key"
    counts_df.columns.name = "gene"

    file_info = pd.DataFrame(file_records)
    return counts_df, file_info

counts_df, file_info = build_counts_matrix(
    cfg.COUNTS_DIR,
    file_glob=cfg.COUNT_FILE_GLOB,
    recursive=cfg.RECURSIVE_SEARCH,
)

print("counts matrix shape:", counts_df.shape)
display(file_info.head())
display(counts_df.iloc[:3, :8])


In [ ]:
def prepare_metadata(meta_df: pd.DataFrame, diagnoses: Iterable[str], visits: Iterable[str], qc_pass_only: bool = True) -> pd.DataFrame:
    meta = meta_df.copy()

    for col in ["PATNO", "DIAGNOSIS", "CLINICAL_EVENT", "QCflagIR3", "HudAlphaID"]:
        if col in meta.columns:
            meta[col] = meta[col].astype(str)

    # Sample 컬럼을 기준으로 counts 파일명과 직접 매칭
    if "Sample" not in meta.columns:
        raise ValueError("Metadata must contain a 'Sample' column for sample-name matching.")

    meta["normalized_sample_key"] = meta["Sample"].map(normalize_sample_key)

    if "DIAGNOSES" not in meta.columns and "DIAGNOSIS" not in meta.columns:
        raise ValueError("Metadata must contain 'DIAGNOSIS' column.")

    if "CLINICAL_EVENT" not in meta.columns:
        raise ValueError("Metadata must contain 'CLINICAL_EVENT' column.")

    meta = meta[meta["DIAGNOSIS"].isin(list(diagnoses))]
    meta = meta[meta["CLINICAL_EVENT"].isin(list(visits))]

    if qc_pass_only:
        if "QCflagIR3" not in meta.columns:
            raise ValueError("cfg.QC_PASS_ONLY=True 이지만 metadata에 'QCflagIR3' 컬럼이 없습니다.")
        meta = meta[meta["QCflagIR3"].str.lower().eq("pass")]

    return meta.reset_index(drop=True)

def match_counts_to_metadata(counts_df: pd.DataFrame, file_info: pd.DataFrame, meta_df: pd.DataFrame):
    meta = meta_df.copy()
    files = file_info.copy()

    files["sample_key"] = files["sample_key"].astype(str)
    files["HudAlphaID_file"] = files["HudAlphaID_file"].astype(str)
    meta["normalized_sample_key"] = meta["normalized_sample_key"].astype(str)
    if "HudAlphaID" in meta.columns:
        meta["HudAlphaID"] = meta["HudAlphaID"].astype(str)

    # 1차: Sample 이름 exact match
    files = files.set_index("sample_key").loc[counts_df.index].reset_index()
    exact = files.merge(
        meta,
        left_on="sample_key",
        right_on="normalized_sample_key",
        how="left",
        suffixes=("", "_meta"),
    )

    # 2차: HudAlphaID fallback
    if "HudAlphaID" in meta.columns:
        need_fallback = exact["PATNO"].isna()
        if need_fallback.any():
            fallback = files.loc[need_fallback, ["sample_key", "HudAlphaID_file"]].merge(
                meta,
                left_on="HudAlphaID_file",
                right_on="HudAlphaID",
                how="left",
                suffixes=("", "_meta"),
            )
            fallback = fallback.set_index("sample_key")
            exact = exact.set_index("sample_key")
            fill_cols = [c for c in fallback.columns if c in exact.columns]
            for col in fill_cols:
                exact.loc[fallback.index, col] = exact.loc[fallback.index, col].combine_first(fallback[col])
            exact = exact.reset_index()

    matched = exact[~exact["PATNO"].isna()].copy()
    if matched.empty:
        raise ValueError("No count files were matched to metadata.")

    matched["analysis_sample_id"] = matched["sample_key"].astype(str)
    matched = matched.set_index("analysis_sample_id")

    counts_matched = counts_df.loc[matched["sample_key"]].copy()
    counts_matched.index = matched.index

    metadata_matched = matched.copy()
    metadata_matched["condition"] = metadata_matched["DIAGNOSIS"].astype(str)

    return counts_matched, metadata_matched

meta_filtered = prepare_metadata(
    meta_raw,
    diagnoses=cfg.DIAGNOSES,
    visits=cfg.VISITS,
    qc_pass_only=cfg.QC_PASS_ONLY,
)

counts_matched, metadata_matched = match_counts_to_metadata(counts_df, file_info, meta_filtered)

print("filtered metadata shape:", meta_filtered.shape)
print("matched counts shape:", counts_matched.shape)
print("matched metadata shape:", metadata_matched.shape)
print("\ncondition counts:")
print(metadata_matched["condition"].value_counts())

display(metadata_matched.head())


In [ ]:
def dedup_keep_order(items: Iterable[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def get_design_columns(cfg: AnalysisConfig) -> Tuple[List[str], List[str], List[str], List[str]]:
    categorical = list(cfg.CATEGORICAL_COVARIATES)
    continuous = list(cfg.CONTINUOUS_COVARIATES)
    design_factors = dedup_keep_order(categorical + continuous + [cfg.CONDITION_COL])
    required = dedup_keep_order(design_factors)
    return required, categorical, continuous, design_factors

def finalize_design_metadata(metadata_matched: pd.DataFrame, cfg: AnalysisConfig) -> pd.DataFrame:
    meta = metadata_matched.copy()
    required, categorical, continuous, design_factors = get_design_columns(cfg)

    missing_cols = [c for c in required if c not in meta.columns]
    if missing_cols:
        raise ValueError(f"Missing design/covariate columns in metadata: {missing_cols}")

    for c in categorical + [cfg.CONDITION_COL]:
        if c in meta.columns:
            meta[c] = meta[c].astype(str)

    for c in continuous:
        meta[c] = pd.to_numeric(meta[c], errors="coerce")

    before = meta.shape[0]
    meta = meta.dropna(subset=required).copy()
    after = meta.shape[0]
    print(f"Dropped {before - after} samples due to NA in design columns: {required}")

    return meta

metadata_final = finalize_design_metadata(metadata_matched, cfg)
counts_final = counts_matched.loc[metadata_final.index].copy()

required, categorical, continuous, design_factors = get_design_columns(cfg)
print("Required design columns:", required)
print("Categorical covariates:", categorical)
print("Continuous covariates:", continuous)
print("Design factors:", design_factors)
print("\nFinal condition counts:")
print(metadata_final[cfg.CONDITION_COL].value_counts())

display(metadata_final[required + ["PATNO", "CLINICAL_EVENT"]].head())


In [ ]:
# 필요 시 여기서 설정을 다시 바꾼 뒤 아래 셀부터 재실행하세요.
# 예:
# cfg.VISITS = ("BL", "V04", "V06", "V08")
# cfg.CATEGORICAL_COVARIATES = ("GENDER",)
# cfg.CONTINUOUS_COVARIATES = ("AGE",)
# cfg.N_SPLITS = 5
# cfg.SIGNIFICANCE_METRIC = "padj"
# cfg.ALPHA = 0.05


In [ ]:
def filter_genes_by_total_count(counts_df: pd.DataFrame, min_total_count_per_gene: int) -> pd.DataFrame:
    keep = counts_df.sum(axis=0) >= int(min_total_count_per_gene)
    return counts_df.loc[:, keep]

def validate_subset_for_de(meta_df: pd.DataFrame, cfg: AnalysisConfig) -> None:
    vc = meta_df[cfg.CONDITION_COL].astype(str).value_counts()
    if cfg.CASE_LABEL not in vc.index or cfg.CONTROL_LABEL not in vc.index:
        raise ValueError(f"Both labels must be present. Got counts: {vc.to_dict()}")
    if (vc[[cfg.CASE_LABEL, cfg.CONTROL_LABEL]] < cfg.MIN_SAMPLES_PER_GROUP_FOR_DE).any():
        raise ValueError(
            f"Each group must have at least {cfg.MIN_SAMPLES_PER_GROUP_FOR_DE} samples. "
            f"Got counts: {vc.to_dict()}"
        )

def run_single_deseq(counts_df: pd.DataFrame, meta_df: pd.DataFrame, cfg: AnalysisConfig) -> Dict[str, object]:
    validate_subset_for_de(meta_df, cfg)

    counts_use = filter_genes_by_total_count(counts_df, cfg.MIN_TOTAL_COUNT_PER_GENE).copy().astype(int)
    meta_use = meta_df.copy()

    required, categorical, continuous, design_factors = get_design_columns(cfg)

    for c in categorical + [cfg.CONDITION_COL]:
        if c in meta_use.columns:
            meta_use[c] = meta_use[c].astype(str)

    for c in continuous:
        meta_use[c] = pd.to_numeric(meta_use[c], errors="coerce")

    inference = DefaultInference(n_cpus=cfg.N_CPUS)

    dds = DeseqDataSet(
        counts=counts_use,
        metadata=meta_use,
        design_factors=design_factors,
        continuous_factors=continuous if len(continuous) > 0 else None,
        ref_level=[cfg.CONDITION_COL, cfg.CONTROL_LABEL],
        fit_type=cfg.FIT_TYPE,
        refit_cooks=cfg.REFIT_COOKS,
        min_replicates=cfg.MIN_REPLICATES,
        n_cpus=cfg.N_CPUS,
        inference=inference,
        quiet=cfg.QUIET,
        low_memory=cfg.LOW_MEMORY,
    )
    dds.deseq2()

    stat_res = DeseqStats(
        dds,
        contrast=[cfg.CONDITION_COL, cfg.CASE_LABEL, cfg.CONTROL_LABEL],
        alpha=cfg.ALPHA,
        cooks_filter=cfg.COOKS_FILTER,
        independent_filter=cfg.INDEPENDENT_FILTER,
        inference=inference,
        quiet=cfg.QUIET,
    )
    stat_res.summary()

    res = stat_res.results_df.copy()
    res.index.name = "gene"
    res = res.reset_index()

    metric = cfg.SIGNIFICANCE_METRIC
    if metric not in {"pvalue", "padj"}:
        raise ValueError("SIGNIFICANCE_METRIC must be either 'pvalue' or 'padj'.")

    res["is_significant"] = res[metric].notna() & (res[metric] < cfg.ALPHA)

    return {
        "dds": dds,
        "stat_res": stat_res,
        "results_df": res,
        "n_samples": meta_use.shape[0],
        "n_genes_input": counts_df.shape[1],
        "n_genes_tested": counts_use.shape[1],
        "group_counts": meta_use[cfg.CONDITION_COL].value_counts().to_dict(),
        "design_factors": design_factors,
        "continuous_factors": continuous,
    }


In [ ]:
full_out = run_single_deseq(counts_final, metadata_final, cfg)
full_results_df = full_out["results_df"].copy()

print("Full dataset DE complete")
print("n_samples:", full_out["n_samples"])
print("group_counts:", full_out["group_counts"])
print("n_genes_input:", full_out["n_genes_input"])
print("n_genes_tested:", full_out["n_genes_tested"])
print("n_significant:", int(full_results_df["is_significant"].sum()))

display(full_results_df.sort_values(cfg.SIGNIFICANCE_METRIC).head(20))


In [ ]:
def make_kfold_train_test_subsets(counts_df: pd.DataFrame, meta_df: pd.DataFrame, cfg: AnalysisConfig):
    y = meta_df[cfg.CONDITION_COL].astype(str).values
    sample_ids = meta_df.index.to_numpy()

    skf = StratifiedKFold(
        n_splits=cfg.N_SPLITS,
        shuffle=cfg.SHUFFLE,
        random_state=cfg.RANDOM_STATE,
    )

    for fold, (train_idx, test_idx) in enumerate(skf.split(sample_ids, y), start=1):
        train_ids = sample_ids[train_idx].tolist()
        test_ids = sample_ids[test_idx].tolist()

        yield {
            "fold": fold,
            "subset": "train",
            "sample_ids": train_ids,
            "counts": counts_df.loc[train_ids].copy(),
            "metadata": meta_df.loc[train_ids].copy(),
        }
        yield {
            "fold": fold,
            "subset": "test",
            "sample_ids": test_ids,
            "counts": counts_df.loc[test_ids].copy(),
            "metadata": meta_df.loc[test_ids].copy(),
        }

subsets_preview = list(make_kfold_train_test_subsets(counts_final, metadata_final, cfg))
preview_rows = []
for item in subsets_preview:
    vc = item["metadata"][cfg.CONDITION_COL].value_counts().to_dict()
    preview_rows.append({
        "fold": item["fold"],
        "subset": item["subset"],
        "n_samples": item["metadata"].shape[0],
        "group_counts": vc,
    })
preview_df = pd.DataFrame(preview_rows)
display(preview_df)


In [ ]:
def run_kfold_train_test_de(counts_df: pd.DataFrame, meta_df: pd.DataFrame, cfg: AnalysisConfig):
    run_summaries = []
    all_results = []
    errors = []

    for item in make_kfold_train_test_subsets(counts_df, meta_df, cfg):
        fold = item["fold"]
        subset = item["subset"]
        run_id = f"fold{fold}_{subset}"

        try:
            out = run_single_deseq(item["counts"], item["metadata"], cfg)
            res = out["results_df"].copy()
            res["fold"] = fold
            res["subset"] = subset
            res["run_id"] = run_id
            res["n_samples"] = out["n_samples"]
            res["n_genes_input"] = out["n_genes_input"]
            res["n_genes_tested"] = out["n_genes_tested"]
            res["n_case"] = out["group_counts"].get(cfg.CASE_LABEL, 0)
            res["n_control"] = out["group_counts"].get(cfg.CONTROL_LABEL, 0)

            all_results.append(res)

            run_summaries.append({
                "run_id": run_id,
                "fold": fold,
                "subset": subset,
                "status": "ok",
                "n_samples": out["n_samples"],
                "n_case": out["group_counts"].get(cfg.CASE_LABEL, 0),
                "n_control": out["group_counts"].get(cfg.CONTROL_LABEL, 0),
                "n_genes_input": out["n_genes_input"],
                "n_genes_tested": out["n_genes_tested"],
                "n_significant": int(res["is_significant"].sum()),
                "median_abs_log2fc": float(res["log2FoldChange"].abs().median(skipna=True)),
            })
        except Exception as e:
            errors.append({
                "run_id": run_id,
                "fold": fold,
                "subset": subset,
                "error": str(e),
            })
            run_summaries.append({
                "run_id": run_id,
                "fold": fold,
                "subset": subset,
                "status": "error",
                "error": str(e),
            })

    run_summary_df = pd.DataFrame(run_summaries)
    all_results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    errors_df = pd.DataFrame(errors)

    return run_summary_df, all_results_df, errors_df

run_summary_df, kfold_results_df, errors_df = run_kfold_train_test_de(counts_final, metadata_final, cfg)

print("K-fold train/test DE finished")
display(run_summary_df)

if not errors_df.empty:
    print("\nErrors:")
    display(errors_df)


In [ ]:
def get_sig_gene_set(df: pd.DataFrame) -> set:
    if df.empty:
        return set()
    return set(df.loc[df["is_significant"], "gene"].astype(str))

def compare_run_to_full(run_df: pd.DataFrame, full_df: pd.DataFrame) -> Dict[str, float]:
    full_sig = get_sig_gene_set(full_df)
    run_sig = get_sig_gene_set(run_df)

    overlap = full_sig & run_sig
    union = full_sig | run_sig

    merged = full_df[["gene", "log2FoldChange"]].merge(
        run_df[["gene", "log2FoldChange"]],
        on="gene",
        how="inner",
        suffixes=("_full", "_run")
    )

    if merged.shape[0] >= 2:
        log2fc_corr = merged["log2FoldChange_full"].corr(merged["log2FoldChange_run"], method="spearman")
        direction_match = (
            np.sign(merged["log2FoldChange_full"].fillna(0)) ==
            np.sign(merged["log2FoldChange_run"].fillna(0))
        ).mean()
    else:
        log2fc_corr = np.nan
        direction_match = np.nan

    return {
        "n_sig_full": len(full_sig),
        "n_sig_run": len(run_sig),
        "n_overlap_sig": len(overlap),
        "jaccard_sig": (len(overlap) / len(union)) if len(union) > 0 else np.nan,
        "recall_vs_full_sig": (len(overlap) / len(full_sig)) if len(full_sig) > 0 else np.nan,
        "precision_vs_full_sig": (len(overlap) / len(run_sig)) if len(run_sig) > 0 else np.nan,
        "n_common_genes_for_lfc": merged.shape[0],
        "spearman_log2fc_vs_full": log2fc_corr,
        "direction_match_rate_vs_full": direction_match,
    }

comparison_rows = []
for run_id, run_df in kfold_results_df.groupby("run_id"):
    meta = run_df[["fold", "subset"]].drop_duplicates().iloc[0].to_dict()
    stats = compare_run_to_full(run_df, full_results_df)
    comparison_rows.append({
        "run_id": run_id,
        "fold": int(meta["fold"]),
        "subset": meta["subset"],
        **stats,
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(["fold", "subset"]).reset_index(drop=True)
display(comparison_df)


In [ ]:
def summarize_gene_stability(kfold_results_df: pd.DataFrame) -> pd.DataFrame:
    if kfold_results_df.empty:
        return pd.DataFrame()

    grouped = kfold_results_df.groupby(["subset", "gene"], dropna=False)
    summary = grouped.agg(
        runs_present=("run_id", "nunique"),
        sig_frequency=("is_significant", "mean"),
        mean_log2fc=("log2FoldChange", "mean"),
        std_log2fc=("log2FoldChange", "std"),
        median_log2fc=("log2FoldChange", "median"),
        mean_padj=("padj", "mean"),
        min_padj=("padj", "min"),
    ).reset_index()

    return summary.sort_values(["subset", "sig_frequency", "min_padj"], ascending=[True, False, True])

gene_stability_df = summarize_gene_stability(kfold_results_df)
display(gene_stability_df.head(20))


In [ ]:
# full과 각 run의 significance 비교용 heatmap-like pivot 테이블
if not kfold_results_df.empty:
    sig_pivot = (
        kfold_results_df[["gene", "run_id", "is_significant"]]
        .assign(is_significant=lambda x: x["is_significant"].astype(int))
        .pivot(index="gene", columns="run_id", values="is_significant")
        .fillna(0)
        .astype(int)
    )

    full_sig_series = (
        full_results_df[["gene", "is_significant"]]
        .assign(full_significant=lambda x: x["is_significant"].astype(int))
        .set_index("gene")["full_significant"]
    )

    sig_pivot = sig_pivot.join(full_sig_series, how="left").fillna(0).astype(int)
    sig_pivot["sig_count_across_runs"] = sig_pivot.drop(columns=["full_significant"]).sum(axis=1)

    display(sig_pivot.sort_values(["full_significant", "sig_count_across_runs"], ascending=[False, False]).head(30))


In [ ]:
# 간단한 시각화 1: run별 유의 DEG 수
if not run_summary_df.empty:
    plot_df = run_summary_df[run_summary_df["status"] == "ok"].copy()
    plt.figure()
    plt.bar(plot_df["run_id"], plot_df["n_significant"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Number of significant DEGs")
    plt.title("Significant DEGs per fold subset")
    plt.tight_layout()
    plt.show()


In [ ]:
# 간단한 시각화 2: full 대비 Jaccard
if not comparison_df.empty:
    plt.figure()
    plt.bar(comparison_df["run_id"], comparison_df["jaccard_sig"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Jaccard with full significant DEG set")
    plt.title("Overlap with full DE result")
    plt.tight_layout()
    plt.show()


In [ ]:
# 간단한 시각화 3: full 대비 log2FC Spearman correlation
if not comparison_df.empty:
    plt.figure()
    plt.bar(comparison_df["run_id"], comparison_df["spearman_log2fc_vs_full"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Spearman correlation")
    plt.title("log2FC correlation vs full dataset")
    plt.tight_layout()
    plt.show()


In [ ]:
def attach_full_result_info_to_stability(gene_stability_df: pd.DataFrame, full_results_df: pd.DataFrame) -> pd.DataFrame:
    full_info = full_results_df[[
        "gene", "baseMean", "log2FoldChange", "lfcSE", "stat", "pvalue", "padj", "is_significant"
    ]].copy()
    full_info = full_info.rename(columns={
        "baseMean": "full_baseMean",
        "log2FoldChange": "full_log2FoldChange",
        "lfcSE": "full_lfcSE",
        "stat": "full_stat",
        "pvalue": "full_pvalue",
        "padj": "full_padj",
        "is_significant": "full_is_significant",
    })
    out = gene_stability_df.merge(full_info, on="gene", how="left")
    return out

gene_stability_with_full_df = attach_full_result_info_to_stability(gene_stability_df, full_results_df)

display(
    gene_stability_with_full_df.sort_values(
        ["subset", "full_is_significant", "sig_frequency", "full_padj"],
        ascending=[True, False, False, True]
    ).head(30)
)


In [ ]:
# 결과 저장
full_results_path = Path(cfg.RESULTS_DIR) / "full_dataset_deseq_results.csv"
run_summary_path = Path(cfg.RESULTS_DIR) / "kfold_run_summary.csv"
kfold_results_path = Path(cfg.RESULTS_DIR) / "kfold_all_results_long.csv"
comparison_path = Path(cfg.RESULTS_DIR) / "kfold_vs_full_comparison.csv"
gene_stability_path = Path(cfg.RESULTS_DIR) / "kfold_gene_stability.csv"
gene_stability_full_path = Path(cfg.RESULTS_DIR) / "kfold_gene_stability_with_full.csv"
errors_path = Path(cfg.RESULTS_DIR) / "kfold_errors.csv"
config_path = Path(cfg.RESULTS_DIR) / "analysis_config.json"

full_results_df.to_csv(full_results_path, index=False)
run_summary_df.to_csv(run_summary_path, index=False)
kfold_results_df.to_csv(kfold_results_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
gene_stability_df.to_csv(gene_stability_path, index=False)
gene_stability_with_full_df.to_csv(gene_stability_full_path, index=False)
errors_df.to_csv(errors_path, index=False)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)

print("Saved files:")
for p in [
    full_results_path,
    run_summary_path,
    kfold_results_path,
    comparison_path,
    gene_stability_path,
    gene_stability_full_path,
    errors_path,
    config_path,
]:
    print("-", p)


# 결과 해석 가이드

## 1) `full_dataset_deseq_results.csv`
전체 데이터셋에 대해 수행한 기준 DE 결과입니다.

주요 컬럼:
- `gene`: 유전자 ID
- `baseMean`: 정규화된 평균 발현량
- `log2FoldChange`: case(`cfg.CASE_LABEL`) vs control(`cfg.CONTROL_LABEL`)의 효과 크기
- `pvalue`, `padj`: 유의성
- `is_significant`: 현재 기준(`SIGNIFICANCE_METRIC < ALPHA`)에서 유의 여부

---

## 2) `kfold_run_summary.csv`
각 fold의 train/test subset에 대해 DE가 성공했는지와 요약 통계가 들어 있습니다.

주요 컬럼:
- `run_id`: 예) `fold1_train`, `fold1_test`
- `n_case`, `n_control`: 각 subset의 군별 표본 수
- `n_significant`: 유의 DEG 수

이 표를 보면 subset 크기 차이 때문에 train/test의 DEG 수가 얼마나 달라지는지 볼 수 있습니다.

---

## 3) `kfold_vs_full_comparison.csv`
각 run 결과를 full 결과와 직접 비교한 표입니다.

핵심 지표:
- `n_overlap_sig`: full과 run에서 모두 유의한 유전자 수
- `jaccard_sig`: 두 유의 DEG 집합의 Jaccard 유사도
- `recall_vs_full_sig`: full 유의 DEG 중 run이 다시 찾아낸 비율
- `precision_vs_full_sig`: run 유의 DEG 중 full과 겹치는 비율
- `spearman_log2fc_vs_full`: full과 run 간 log2FC 순위 상관
- `direction_match_rate_vs_full`: 효과 방향(증가/감소) 일치율

---

## 4) `kfold_gene_stability.csv`
각 유전자가 fold 반복에서 얼마나 안정적으로 유의하게 나오는지를 정리합니다.

핵심 컬럼:
- `subset`: `train` 또는 `test`
- `runs_present`: 해당 subset 계열 run들에 등장한 횟수
- `sig_frequency`: 유의하게 나온 비율
- `mean_log2fc`, `std_log2fc`: 효과 크기 평균과 변동성

예:
- `sig_frequency`가 높고 `std_log2fc`가 낮으면 상대적으로 안정적인 gene
- `sig_frequency`가 낮고 `std_log2fc`가 크면 분할에 민감한 gene

---

## 5) `kfold_gene_stability_with_full.csv`
위 안정성 표에 full 결과를 함께 붙인 버전입니다.

이 파일로 다음 질문을 볼 수 있습니다:
- full에서는 유의했는데 fold test에서는 자주 사라지는가?
- train에서는 안정적인데 test에서는 불안정한가?
- full과 subset 간 효과 방향은 일관적인가?

---

## 권장 해석 순서
1. `kfold_run_summary.csv`에서 train/test별 DEG 수 변동 확인
2. `kfold_vs_full_comparison.csv`에서 full과의 유사도 확인
3. `kfold_gene_stability_with_full.csv`에서 안정적/불안정 gene 확인
4. 필요 시 특정 gene만 골라 boxplot 또는 rank comparison 추가

---

## 확장 아이디어
- 방문시점 여러 개를 동시에 포함: `cfg.VISITS = ("BL", "V04", "V06", "V08")`
- 공변량 보정:
  - 예: `cfg.CATEGORICAL_COVARIATES = ("GENDER",)`
  - 예: `cfg.CONTINUOUS_COVARIATES = ("AGE",)`
- diagnosis 조합 변경:
  - 예: `("PD", "SWEDD")`
- patient-level leakage 방지가 필요하면, 향후 `PATNO` 기준 GroupKFold 버전으로 확장 가능
